# 01 Bronze Layer
**EAS 587 Phase 3 | Drug Overdose CDC Dataset**



| Table | Source File |
|---|---|
| `bronze_cdc_overdose` | `VSRR_Provisional_Drug_Overdose_Death_Counts_20260214.csv` - CDC provisional overdose death counts |
| `bronze_kff_opioid_disorder` | `kff_opioid_disorder_by_state.csv` - KFF/NSDUH opioid use disorder rates by state |

In [0]:
#  CONFIGs

CDC_PATH = "/Volumes/workspace/default/p3_data_raw/VSRR_Provisional_Drug_Overdose_Death_Counts_20260214.csv"
KFF_PATH = "/Volumes/workspace/default/p3_data_raw/kff_opioid_additional_data.csv"
DB_NAME  = "eas587_phase3"


spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
spark.sql(f"USE {DB_NAME}")
print(f"Using database: {DB_NAME}")

Using database: eas587_phase3


## Bronze 1 — CDC Drug Overdose Deaths

In [0]:

bronze_cdc = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false") #need to preserve exact schema , few columns had whitespace
    .csv(CDC_PATH)
)

print(f"Columns : {bronze_cdc.columns}")
print(f"Rows    : {bronze_cdc.count():,}")
bronze_cdc.show(5, truncate=False)

Columns : ['State', 'Year', 'Month', 'Period', 'Indicator', 'Data Value', 'Percent Complete', 'Percent Pending Investigation', 'State Name', 'Footnote', 'Footnote Symbol', 'Predicted Value']
Rows    : 81,270
+-----+----+--------+---------------+---------------+----------+----------------+-----------------------------+----------+------------------------------------------------------------------------------------------------------------------------+---------------+---------------+
|State|Year|Month   |Period         |Indicator      |Data Value|Percent Complete|Percent Pending Investigation|State Name|Footnote                                                                                                                |Footnote Symbol|Predicted Value|
+-----+----+--------+---------------+---------------+----------+----------------+-----------------------------+----------+----------------------------------------------------------------------------------------------------------------------

In [0]:
(
    bronze_cdc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB_NAME}.bronze_cdc_overdose")
)
print("✅ bronze_cdc_overdose written")
spark.sql(f"SELECT COUNT(*) AS rows FROM {DB_NAME}.bronze_cdc_overdose").show()

✅ bronze_cdc_overdose written
+-----+
| rows|
+-----+
|81270|
+-----+



## Bronze 2: KFF Opioid Use Disorder by State

Source: SAMHSA NSDUH 2022-2023 via Kaiser Family Foundation  
Contains: share of adolescents (12-17) and adults (18+) with past-year opioid use disorder, by state.



In [0]:

bronze_kff = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # keep as strings; Silver casts
    .csv(KFF_PATH)
)

print(f"Columns : {bronze_kff.columns}")
print(f"Rows    : {bronze_kff.count():,}")
bronze_kff.show(truncate=False)

Columns : ['Location', 'Adolescents Ages 12-17 with an Opioid Use Disorder', 'Adults Ages 18+ with an Opioid Use Disorder']
Rows    : 62
+--------------------+--------------------------------------------------+-------------------------------------------+
|Location            |Adolescents Ages 12-17 with an Opioid Use Disorder|Adults Ages 18+ with an Opioid Use Disorder|
+--------------------+--------------------------------------------------+-------------------------------------------+
|Alabama             |0.01                                              |0.027                                      |
|Alaska              |0.008                                             |0.022                                      |
|Arizona             |0.013                                             |0.024                                      |
|Arkansas            |0.014                                             |0.023                                      |
|California          |0.011          

In [0]:
# Rename columns to remove invalid characters for Delta Lake only whitespace special chr is allowed

bronze_kff_clean = bronze_kff.toDF(
    "Location",
    "Adolescents_12_17_Opioid_Use_Disorder",
    "Adults_18_Plus_Opioid_Use_Disorder"
)

(
    bronze_kff_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB_NAME}.bronze_kff_opioid_disorder")
)
print("✅ bronze_kff_opioid_disorder written")
spark.sql(f"SELECT COUNT(*) AS rows FROM {DB_NAME}.bronze_kff_opioid_disorder").show()

✅ bronze_kff_opioid_disorder written
+----+
|rows|
+----+
|  62|
+----+

